# Проект 1. Аналитическая записка

Автор: Ирина Чернова (idchernova)

Перед сдачей: Kernel → Restart Kernel and Run All Cells, всё должно пройти без ошибок.

## 0. Бизнес-вопрос и гипотезы

1 марта 2026 года «Базовый» подорожал с 199 до 249 ₽, «Расширенный» — с 399 до 449 ₽.

Вопрос: …

Гипотезы:
- А1. Рост выручки после 1 марта дала цена, а не число подписчиков.
- А2. Средний чек вырос ещё и потому, что клиенты перешли на «Расширенный».
- А3. Подорожание ускорило отток.
- Б1. Изменение выручки дал один канал привлечения.

Ключевая метрика: выручка от подписок в день, пересчитанная в «нормальный месяц» (× 30,4). Сравниваю январь–февраль 2026 с мартом–июлем 2026.

In [10]:
pip install prime

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for prime from https://files.pythonhosted.org/packages/25/f8/71adbd43a557985fce32c5bbfc74c60b7ff8f971dd292af60d11a199a74d/prime-0.18.0-py3-none-any.whl.metadata
  Obtaining dependency information for boto3>=1.34.0 from https://files.pythonhosted.org/packages/bb/a3/65bb3d64928f3c2da17640b92044084150317f02adf22442cb0c52af2a8b/boto3-1.43.118-py3-none-any.whl.metadata
  Obtaining dependency information for build>=1.0.0 from https://files.pythonhosted.org/packages/ad/9b/9fb3585dabcd73a1b2a6267f63f62649347c9e6d072c9fde365b105abb2c/build-1.6.1-py3-none-any.whl.metadata
  Obtaining dependency information for cryptography>=41.0.0 from https://files.pythonhosted.org/packages/e5/56/d194340cc4a57535e82e1bee9e89667ac4b7c13b5d3f59686deae3094dd5/cryptography-50.0.2-cp311-abi3-macosx_11_0_arm64.whl.metadata
  Installing build dependencies ... done
  Getting requirements to build wheel ... d

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sqlalchemy import text

from prime.config import get_engine

engine = get_engine()


def q(sql: str, **params) -> pd.DataFrame:
    """Выполнить запрос и вернуть DataFrame. Соединение закрывается сразу."""
    try:
        with engine.connect() as conn:
            conn.execute(text("set time zone 'UTC'"))
            return pd.read_sql(text(sql), conn, params=params)
    finally:
        engine.dispose()


def rub(x: float) -> str:
    return f"{x:,.0f} руб.".replace(",", " ")

print("подключились как:", q("select current_user as login").iloc[0, 0])

ModuleNotFoundError: No module named 'prime'

Этот блок описывает первый взгляд на данные - выручка по месяцам и в пересчёте на день. Он показывает, что месяцы надо сравнивать в пересчёте на день, и что единственное настоящее изменение — мартовский скачок

## 1. Данные и зерно расчёта

### 1.1 Выручка по месяцам

Выручка по месяцам как есть и в пересчёте на день. Делю на число дней, за которые в данных есть платежи (days_obs): месяцы разной длины, а 14–15 апреля 2026 платежей нет совсем.

In [ ]:
overview = q("""
    select to_char(paid_at, 'YYYY-MM')        as month,
           count(*)                           as n_payments,
           count(distinct paid_at::date)      as days_obs,
           sum(amount)                        as revenue
    from prime.payments
    where status = 'success'
    group by 1
    order by 1
""")

# сколько дней в месяце по календарю
overview["days_calendar"] = pd.PeriodIndex(overview["month"], freq="M").days_in_month
overview["revenue_per_day"] = overview["revenue"] / overview["days_obs"]
overview["growth_pct"] = overview["revenue"].pct_change() * 100                  
overview["growth_per_day_pct"] = overview["revenue_per_day"].pct_change() * 100   

overview.round(1)

Что тут происходит: берутся из базы все успешные платежи и группируются по месяцам. для каждого месяца считается число платежей, выручка и сколько дней в месяце были платежи. потом добавляется, сколько дней в месяце по календарю, выручка в день (выручка/дни с платежами) и прирост к прошлому месяцу двумя способами (как есть и в пересчёте на день).

Зачем делаем: месяцы разной длины, а подписка списывается раз в 30 дней, поэтому выручку месяцев напрямую сравнивать нельзя. К тому же в апреле 2026 нет данных за 14–15 число. 

### 1.2 Зерно, период, сегмент

Одна строка таблицы monthly — месяц × тариф × канал привлечения: активные подписки, успешные платежи и выручка.

<!-- заметки:
- зерно платёжное: выручка = сумма платежей; если считать по людям, шеринг раздувает выручку (раздел 3)
- тариф: подорожание разное (+25 % и +12,5 %); внутри тарифа цена платежа — чистая цена
- канал: чтобы проверить Б1
- месяц и пересчёт на день: списание раз в 30 дней, месяцы 28–31 день, дыра 14–15.04
- период: база янв–фев 2026, после мар–июл 2026; август не беру — неполный и массовые отмены partner_telecom 22–23.08
- сегмент: вся база; мой сегмент из ДЗ-2 (4 653 клиента, 3 204 подписки) слишком мал для оттока по тарифам
-->

Почему такое зерно: …

Период: …

Сегмент: …

In [ ]:
PRICE_CHANGE = "2026-03-01"                 # день подорожания
BASE = ("2026-01-01", "2026-03-01")         # база: январь–февраль 2026, правый конец не включён
AFTER = ("2026-03-01", "2026-08-01")     # после: март–июль 2026, правый конец не включён
HISTORY = "2025-09-01"                      # начало истории для оттока
DAYS_IN_MONTH = 365.25 / 12                 # средний месяц: «₽ в день» -> «₽ в нормальный месяц»

LAST_MONTH = (pd.Timestamp(AFTER[1]) - pd.DateOffset(months=1)).strftime("%Y-%m-%d")


def period_of(month: str) -> str:
    """Месяц 'YYYY-MM' -> 'база', 'после' или 'история'."""
    if BASE[0][:7] <= month < BASE[1][:7]:
        return "база"
    if AFTER[0][:7] <= month < AFTER[1][:7]:
        return "после"
    return "история"


overview["period"] = overview["month"].map(period_of)
days = overview.groupby("period")["days_obs"].sum()          # дни с данными в каждом периоде

print("база:", BASE, "| после:", AFTER, "| история с:", HISTORY)
print(days.to_string())

In [ ]:
def load_monthly(client_filter: str = "true", first: str = "2025-09-01", last: str = "2025-12-01") -> pd.DataFrame:
    """Месяц × тариф × канал: активные подписки, число платежей, выручка."""
    return q(f"""
        with seg as (
            select client_id, acquisition_channel as channel
            from prime.clients
            where {client_filter}
        ),
        act as (
            select to_char(m.month_start, 'YYYY-MM') as month, s.tariff_id, c.channel, count(*) as active
            from generate_series(cast(:first as timestamptz), cast(:last as timestamptz), interval '1 month') as m(month_start)
            join prime.subscriptions s
              on s.started_at < m.month_start + interval '1 month'
             and (s.ended_at is null or s.ended_at >= m.month_start)
            join seg c on c.client_id = s.owner_client_id
            group by 1, 2, 3
        ),
        rev as (
            select to_char(p.paid_at, 'YYYY-MM') as month, s.tariff_id, c.channel,
                   count(*) as n_payments, sum(p.amount) as revenue
            from prime.payments p
            join prime.subscriptions s on s.subscription_id = p.subscription_id
            join seg c on c.client_id = s.owner_client_id
            where p.status = 'success'
              and p.paid_at >= cast(:first as timestamptz)
              and p.paid_at < cast(:last as timestamptz) + interval '1 month'
            group by 1, 2, 3
        )
        select a.month, a.tariff_id, a.channel, a.active,
               coalesce(r.n_payments, 0) as n_payments, coalesce(r.revenue, 0) as revenue
        from act a
        left join rev r
          on r.month = a.month and r.tariff_id = a.tariff_id and r.channel is not distinct from a.channel
        order by 1, 2, 3
    """, first=first, last=last)


monthly = load_monthly("true", HISTORY, LAST_MONTH)
monthly_raw = monthly.copy()      # нетронутая копия: пропуски канала на месте, пригодится в контролях

# Клиентов без канала не выбрасываю, а оставляю отдельной строкой.
monthly["channel"] = monthly["channel"].fillna("без канала")
monthly["tariff"] = monthly["tariff_id"].map({1: "Базовый", 2: "Расширенный"})
monthly["period"] = monthly["month"].map(period_of)

print(len(monthly), "строк; месяцы:", monthly["month"].min(), "…", monthly["month"].max())
monthly.head(8)

In [ ]:
# Сразу после загрузки смотрю типы колонок и пропуски.
print(monthly.dtypes.to_string())
print()
print("пропусков:", int(monthly.isna().sum().sum()))
print("пропусков канала в сырой копии:", int(monthly_raw["channel"].isna().sum()), "строк")
print("тарифов без названия:", int(monthly["tariff"].isna().sum()))

## 2. Метрики

- выручка в день = выручка месяца / дни с данными; × 30,4 — выручка «нормального месяца»
- активные подписки — подписка началась до конца месяца и не закончилась до его начала
- платежей на подписку в день = платежи / дни с данными / активные
- цена платежа = выручка / платежи, отдельно по тарифу
- доля тарифа = активные подписки тарифа / все активные
- отток = закончились в месяце / активные на начало месяца, без пробных подписок
- доля отмен too_expensive среди отмен с указанной причиной

Внутри тарифа: выручка в день = активные × платежей на подписку в день × цена платежа. Вся выручка — сумма двух тарифов.

<!-- допиши, зачем каждая метрика нужна для вопроса -->

In [ ]:
# Месяц × тариф: каналы сворачиваю.
m = (monthly.groupby(["month", "tariff"], as_index=False)[["active", "n_payments", "revenue"]].sum()
            .merge(overview[["month", "days_obs"]], on="month", how="left", validate="m:1"))

m["per_sub_day"] = m["n_payments"] / m["days_obs"] / m["active"]     # платежей на подписку в день
m["price"] = m["revenue"] / m["n_payments"]                          # цена платежа
m["revenue_per_day"] = m["revenue"] / m["days_obs"]
m["period"] = m["month"].map(period_of)

m.round(4)

In [ ]:
# Доля тарифа среди активных подписок, % — первый взгляд на гипотезу А2.
active = m.pivot_table(index="month", columns="tariff", values="active", aggfunc="sum")
share = active.div(active.sum(axis=1), axis=0) * 100

share.round(1)

## 3. Контроли

До выводов проверяю четыре вещи: не раздувается ли выручка из-за шеринга, сходится ли выручка monthly с прямой суммой платежей, не теряются ли клиенты без канала и какие цены реально списывались.

In [ ]:
# 1. Шеринг: одни и те же успешные платежи окна тремя путями.
share_check = q("""
    with pay as (
        select payment_id, subscription_id, amount
        from prime.payments
        where status = 'success'
          and paid_at >= cast(:a as date) and paid_at < cast(:b as date)
    ),
    users as (                -- все, кто пользуется подпиской: владелец + участники
        select subscription_id, owner_client_id as client_id from prime.subscriptions
        union all
        select subscription_id, client_id from prime.subscription_members
    )
    select (select count(*)                   from pay) as n_payments,
           (select count(distinct payment_id) from pay) as n_payment_ids,
           (select sum(amount)                from pay) as revenue,
           (select count(*)        from pay join prime.subscriptions s using (subscription_id)) as n_via_subs,
           (select sum(pay.amount) from pay join prime.subscriptions s using (subscription_id)) as revenue_via_subs,
           (select count(*)        from pay join users using (subscription_id)) as n_via_users,
           (select sum(pay.amount) from pay join users using (subscription_id)) as revenue_via_users
""", a=BASE[0], b=AFTER[1]).iloc[0]

inflation = share_check["revenue_via_users"] / share_check["revenue"]
sharing_ok = (share_check["n_payments"] == share_check["n_payment_ids"] == share_check["n_via_subs"]
              and abs(share_check["revenue_via_subs"] - share_check["revenue"]) < 0.01)

print(f"платежей {int(share_check['n_payments'])}, через подписку {int(share_check['n_via_subs'])},"
      f" через всех пользователей {int(share_check['n_via_users'])}")
print("выручка по платежам:       ", rub(share_check["revenue"]))
print("выручка «по пользователям»:", rub(share_check["revenue_via_users"]), f"— завышена в {inflation:.2f} раза")

In [ ]:
# 2. ДЕНЬГИ: monthly против прямой суммы платежей — по каждому месяцу.
from_monthly = monthly.groupby("month")["revenue"].sum()
control = overview.set_index("month").loc[from_monthly.index, "revenue"]
money_gap = (from_monthly - control).abs().max()

# 3. Без канала: pivot_table на сырой копии теряет клиентов без канала, на monthly — нет.
lost = monthly_raw.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum").sum()
kept = monthly.pivot_table(index="channel", columns="month", values="revenue", aggfunc="sum").sum()
no_channel_lost = (control - lost).abs().max()
channels_gap = (control - kept).abs().max()

In [ ]:
# 4. Цены. Ищу пересекающиеся интервалы в справочнике тарифов.
tariffs = q("select * from prime.tariffs order by tariff_id, valid_from")
a = tariffs.reset_index(names="row_id")
pairs = a.merge(a, on="tariff_id", suffixes=("_a", "_b"))
overlap = pairs[(pairs["row_id_a"] < pairs["row_id_b"])
                & (pairs["valid_from_a"] <= pairs["valid_to_b"])
                & (pairs["valid_from_b"] <= pairs["valid_to_a"])]

# Какие суммы на самом деле списывались до и после подорожания.
amounts = q("""
    select s.tariff_id,
           case when p.paid_at < cast(:hike as date) then 'до' else 'после' end as period,
           p.amount, count(*) as n_payments
    from prime.payments p
    join prime.subscriptions s on s.subscription_id = p.subscription_id
    where p.status = 'success'
      and p.paid_at >= cast(:a as date) and p.paid_at < cast(:b as date)
    group by 1, 2, 3
    order by 1, 2 desc, 4 desc
""", hike=PRICE_CHANGE, a=BASE[0], b=AFTER[1])

# Какие суммы списывались: до 1 марта ждём только 199 и 399 ₽, после — только 249 и 449 ₽.
prices_before = set(amounts.loc[amounts["period"] == "до", "amount"])
prices_after = set(amounts.loc[amounts["period"] == "после", "amount"])
prices_ok = prices_before == {199, 399} and prices_after == {249, 449}

print(tariffs[["tariff_id", "tariff_name", "monthly_fee", "valid_from", "valid_to"]].to_string(index=False))
print("\nпересекающихся пар интервалов:", len(overlap))
print()
print(amounts.to_string(index=False))

In [ ]:
# Итог проверок: ok, если результат совпал с ожиданием.
print(f"ШЕРИНГ   платёж -> подписка m:1, выручка без раздутия;"
      f" через всех пользователей было бы ×{inflation:.2f}"
      f"   -> {'ok' if sharing_ok else 'ВНИМАНИЕ'}")
print(f"ДЕНЬГИ   monthly против прямой суммы платежей: расхождение {money_gap:.2f} ₽"
      f"   -> {'ok' if money_gap < 0.01 else 'ВНИМАНИЕ'}")
print(f"КАНАЛЫ   без fillna pivot_table потерял бы до {rub(no_channel_lost)} в месяц;"
      f" с «без канала» расхождение {channels_gap:.2f} ₽"
      f"   -> {'ok' if channels_gap < 0.01 else 'ВНИМАНИЕ'}")
print(f"ЦЕНЫ     пересечений в справочнике {len(overlap)} — цену берём из платежей;"
      f" списывалось до {sorted(prices_before)}, после {sorted(prices_after)}"
      f"   -> {'ok' if prices_ok else 'ВНИМАНИЕ: есть суммы не по прайсу'}")

<!-- заметки: при счёте по пользователям выручка выросла бы в 2,77 раза; monthly сходится с платежами до копейки;
без fillna pivot_table терял до 1,3 млн ₽ в месяц; списывались только 199/399 до и 249/449 после -->

Что показали проверки: …

## 4. Факторная декомпозиция

Сначала смотрю, где изменилась выручка (каналы и тарифы), потом раскладываю изменение на множители. Все суммы — в рублях за «нормальный месяц»: выручка периода / дни с данными × 30,4.

### 4.1 Где: каналы и тарифы

In [ ]:
# Выручка «нормального месяца»: выручка периода / дни с данными × 30,4.
# Series делится на Series по совпадающим меткам: база на дни базы, «после» на дни «после».
level = monthly.groupby("period")["revenue"].sum() / days * DAYS_IN_MONTH
total_delta = level["после"] - level["база"]

# Наивное сравнение, как сделали бы финансы: средняя выручка месяца «после» минус «база».
naive = overview.groupby("period")["revenue"].mean()
naive_delta = naive["после"] - naive["база"]

print("выручка нормального месяца:", rub(level["база"]), "->", rub(level["после"]),
      f"({total_delta / level['база']:+.1%})")
print("наивный прирост:           ", rub(naive_delta))

In [ ]:
def where(by: str) -> pd.DataFrame:
    """Аддитивная модель: изменение выручки по частям (каналам или тарифам)."""
    t = monthly.pivot_table(index=by, columns="period", values="revenue", aggfunc="sum")[["база", "после"]]
    t = t / days[["база", "после"]] * DAYS_IN_MONTH      # делится каждая колонка на свои дни
    t["delta"] = t["после"] - t["база"]
    assert abs(t["delta"].sum() - total_delta) < 0.01, f"{by}: части не сходятся с итогом"
    t = t.sort_values("delta")
    t.loc["итого"] = t.sum()
    return t


by_channel = where("channel")
by_tariff = where("tariff")

print("ПО КАНАЛАМ, ₽ в нормальный месяц")
print(by_channel.round().astype(int).to_string())
print("\nПО ТАРИФАМ, ₽ в нормальный месяц")
print(by_tariff.round().astype(int).to_string())

<!-- заметки: выросли все каналы, на 24–30 % каждый; organic больше всех в рублях, потому что самый большой;
оба тарифа выросли; Б1 не подтвердилась -->

Вывод: …

### 4.2 За счёт чего: активные × платежи × цена

Внутри каждого тарифа раскладываю изменение цепными подстановками в порядке активные → платежей на подписку → цена, потом складываю тарифы. Цена идёт последней, поэтому её вклад — (новая цена − старая) × фактические платежи после подорожания.

In [ ]:
# Функция цепных подстановок лежит в src/prime/factors.py.
from prime.factors import contrib


def factors(rows: pd.DataFrame) -> dict:
    """Три множителя для строк одного тарифа за несколько месяцев. Их произведение — выручка в день."""
    by_month = rows.groupby("month")[["active", "n_payments", "revenue"]].sum()   # каналы -> месяц
    n_days = overview.set_index("month").loc[by_month.index, "days_obs"].sum()
    active = by_month["active"].mean()
    return {"active": active,
            "per_sub_day": by_month["n_payments"].sum() / n_days / active,
            "price": by_month["revenue"].sum() / by_month["n_payments"].sum()}


FACTORS = ["active", "per_sub_day", "price"]
LABELS = {"active": "активные подписки", "per_sub_day": "платежей на подписку", "price": "цена платежа"}
TARIFFS = ["Базовый", "Расширенный"]

x0 = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["period"] == "база")]) for t in TARIFFS}
x1 = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["period"] == "после")]) for t in TARIFFS}

print(pd.DataFrame({(t, p): x[t] for t in TARIFFS for p, x in (("база", x0), ("после", x1))})
      .rename(index=LABELS).round(4).to_string())

In [ ]:
def tariff_effects(x0: dict, x1: dict, order: list[str]) -> pd.DataFrame:
    """Влияния факторов, ₽ в нормальный месяц: строка — тариф, колонка — фактор."""
    rows = {}
    for t in TARIFFS:
        rows[t] = contrib(x0[t], x1[t], order)
    return pd.DataFrame(rows).T * DAYS_IN_MONTH


effects_by_tariff = tariff_effects(x0, x1, FACTORS)
effects = effects_by_tariff.sum()      # сложение по тарифам

assert abs(effects.sum() - total_delta) < 0.01, "сумма влияний не равна изменению выручки"

table = effects_by_tariff.rename(columns=LABELS)
table["всего"] = table.sum(axis=1)
table.loc["итого"] = table.sum()
print("₽ в нормальный месяц; январь–февраль -> март–июль 2026; активные -> платежи -> цена")
print(table.round().astype(int).to_string())
print()
print(f"сумма влияний {rub(effects.sum())} — изменение выручки {rub(total_delta)}")
print(f"наивный прирост {rub(naive_delta)}; цене из него принадлежит {rub(effects['price'])}")

In [ ]:
# От порядка подстановок ответ немного меняется, поэтому смотрю все шесть порядков и среднее.
rows = {}
for order in itertools.permutations(FACTORS):
    name = " → ".join(LABELS[f].split()[0] for f in order)
    rows[name] = tariff_effects(x0, x1, list(order)).sum()      # тарифы складываются
orders = pd.DataFrame(rows).T
orders.loc["среднее по порядкам"] = orders.mean()

orders.rename(columns=LABELS).round().astype(int)

In [ ]:
# Каждый месяц «после» против базы: ровный ли эффект цены.
rows = {}
for month in sorted(monthly.loc[monthly["period"] == "после", "month"].unique()):
    x_month = {t: factors(monthly[(monthly["tariff"] == t) & (monthly["month"] == month)]) for t in TARIFFS}
    rows[month] = tariff_effects(x0, x_month, FACTORS).sum()
by_month = pd.DataFrame(rows).T.rename(columns=LABELS)
by_month["изменение"] = by_month.sum(axis=1)
by_month.round().astype(int)

<!-- заметки: цена +9,19 млн ₽ в месяц из +13,09 млн; активные подписки +3,92 млн; платежи на подписку ≈ 0;
при других порядках цена 8,48–9,19 млн; наивное сравнение дало +14,06 млн и всё приписало цене; А1 подтвердилась -->

Вывод: …

### 4.3 Структура тарифов

Средняя цена платежа = сумма по тарифам (доля платежей × цена). Делю её рост на две части: сдвиг долей при старых ценах и рост цен при новых долях.

In [ ]:
# Сколько платежей у каждого тарифа в базе и после; доля — платежи тарифа / все платежи периода.
payments = m.pivot_table(index="tariff", columns="period", values="n_payments", aggfunc="sum")
share0 = payments["база"] / payments["база"].sum()
share1 = payments["после"] / payments["после"].sum()
price0 = pd.Series({t: x0[t]["price"] for t in TARIFFS})
price1 = pd.Series({t: x1[t]["price"] for t in TARIFFS})

avg0, avg1 = (share0 * price0).sum(), (share1 * price1).sum()
by_structure = ((share1 - share0) * price0).sum()      # сначала доли — при старых ценах
by_price = (share1 * (price1 - price0)).sum()          # потом цены — при новых долях
assert abs(by_structure + by_price - (avg1 - avg0)) < 0.01, "сумма влияний не равна изменению"

print(pd.DataFrame({"доля платежей, база": share0, "доля платежей, после": share1,
                    "цена, база": price0, "цена, после": price1}).round(4).to_string())
print()
print(f"средняя цена платежа: {avg0:.2f} -> {avg1:.2f} ₽ ({avg1 - avg0:+.2f} ₽)")
print(f"  за счёт структуры тарифов: {by_structure:+.2f} ₽")
print(f"  за счёт цен:               {by_price:+.2f} ₽")

<!-- заметки: доли 62 % / 38 % не изменились; из +50,08 ₽ роста средней цены на структуру приходится +0,08 ₽; А2 не подтвердилась -->

Вывод: …

### 4.4 Отток

Отток по месяцам и тарифам — доля подписок, активных на начало месяца, которые закончились в этом месяце. Пробные подписки не беру. «Базовый» подорожал на 25 %, «Расширенный» на 12,5 %, поэтому смотрю, у какого тарифа отток вырос сильнее, и сравниваю разницу с погрешностью.

In [ ]:
subs = q("""
    select subscription_id, tariff_id, started_at, ended_at, cancel_reason
    from prime.subscriptions
    where not is_trial
    order by subscription_id
""")
subs["tariff"] = subs["tariff_id"].map({1: "Базовый", 2: "Расширенный"})


def churn_counts(subs: pd.DataFrame, start: str, end: str) -> tuple[int, int, int, int]:
    """Активные на start, сколько из них закончилось до end, у скольких есть причина,
    у скольких причина too_expensive."""
    start, end = pd.Timestamp(start, tz="UTC"), pd.Timestamp(end, tz="UTC")
    active = subs[(subs["started_at"] < start)
                  & (subs["ended_at"].isna() | (subs["ended_at"] >= start))]
    ended = active[active["ended_at"] < end]
    with_reason = int(ended["cancel_reason"].notna().sum())
    too_expensive = int((ended["cancel_reason"] == "too_expensive").sum())
    return len(active), len(ended), with_reason, too_expensive


month_starts = pd.date_range(HISTORY, AFTER[1], freq="MS").strftime("%Y-%m-%d")
rows = []
for start, end in zip(month_starts[:-1], month_starts[1:]):
    for tariff, g in subs.groupby("tariff"):
        active_start, ended, with_reason, too_expensive = churn_counts(g, start, end)
        rows.append({"month": start[:7], "tariff": tariff, "active_start": active_start, "ended": ended,
                     "with_reason": with_reason, "too_expensive": too_expensive})

churn = pd.DataFrame(rows)
churn["churn"] = churn["ended"] / churn["active_start"]
churn["period"] = np.where(churn["month"] < PRICE_CHANGE[:7], "до", "после")

churn.pivot_table(index="month", columns="tariff", values="churn", aggfunc="sum").mul(100).round(2)

In [ ]:
# До и после — по тарифам. Отток периода = все закончившиеся / все активные на начало месяцев.
pooled = churn.groupby(["tariff", "period"], as_index=False)[["active_start", "ended", "with_reason", "too_expensive"]].sum()
pooled["churn"] = pooled["ended"] / pooled["active_start"]
pooled["share_too_expensive"] = pooled["too_expensive"] / pooled["with_reason"]
# Погрешность доли — стандартная ошибка √(p(1−p)/n).
# Приближение: месяцы одной подписки считаем независимыми.
pooled["se"] = np.sqrt(pooled["churn"] * (1 - pooled["churn"]) / pooled["active_start"])
print(pooled.round(4).to_string(index=False))
print()

change, change_se = {}, {}
for t in TARIFFS:
    before = pooled[(pooled["tariff"] == t) & (pooled["period"] == "до")].iloc[0]
    after = pooled[(pooled["tariff"] == t) & (pooled["period"] == "после")].iloc[0]
    change[t] = after["churn"] - before["churn"]                       # прирост оттока
    change_se[t] = np.sqrt(before["se"] ** 2 + after["se"] ** 2)
    print(f"{t}: отток {change[t] * 100:+.2f} п.п. ± {1.96 * change_se[t] * 100:.2f}")

# Разница приростов: что изменилось у «Базового» сверх того, что изменилось у «Расширенного».
did = change["Базовый"] - change["Расширенный"]
did_ci = 1.96 * np.sqrt(change_se["Базовый"] ** 2 + change_se["Расширенный"] ** 2)
print(f"\nразница приростов («Базовый» минус «Расширенный»): {did * 100:+.2f} п.п. ± {did_ci * 100:.2f}")

<!-- заметки: отток ~3,9–4,0 % в месяц до и после у обоих тарифов; «Базовый» −0,06 п.п. ± 0,08, «Расширенный» 0,00 ± 0,10;
разница −0,06 ± 0,13; too_expensive 24,2 % → 24,2 %; отмен 5–6 тыс. в месяц — эффект от 0,1 п.п. был бы виден; А3 не подтвердилась -->

Вывод: …

## 5. Графики

In [ ]:
ACCENT, GRAY, GRID = "#C1272D", "#9CA3AF", "#E5E7EB"


def style(ax: plt.Axes, title: str, ylabel: str, caption: str) -> None:
    """Оформление графика: заголовок слева, подписи осей, без лишних рамок."""
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel)
    ax.set_xlabel(caption, fontsize=9, color="#6B7280")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color=GRID)
    ax.set_axisbelow(True)

### График 1. Выручка в день по месяцам

In [ ]:
g = overview[overview["month"] >= HISTORY[:7]].copy()
g = g[g["month"] < AFTER[1][:7]]
pre = g.loc[g["month"] < PRICE_CHANGE[:7], "growth_per_day_pct"].mean()
post = g.loc[g["month"] > PRICE_CHANGE[:7], "growth_per_day_pct"].mean()
jump = g.loc[g["month"] == PRICE_CHANGE[:7], "growth_per_day_pct"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
before_hike = g["month"] < PRICE_CHANGE[:7]
ax.plot(g["month"], g["revenue_per_day"] / 1e6, color=GRAY, marker="o", zorder=2)
ax.plot(g.loc[~before_hike, "month"], g.loc[~before_hike, "revenue_per_day"] / 1e6, color=ACCENT,
        marker="o", linewidth=2.5, zorder=3)
ax.axvline(before_hike.sum() - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.annotate("подорожание 1 марта", (before_hike.sum() - 0.45, ax.get_ylim()[0]), fontsize=9,
            xytext=(0, 5), textcoords="offset points")
style(ax, f"В марте выручка в день выросла на {jump:.0f} %; до и после — примерно "
          f"{pre:+.1f} % и {post:+.1f} % в месяц",
      "млн ₽ в день",
      "Месяц по UTC. Выручка месяца / дни с данными: апрель без 14–15 числа. Источник: prime.payments, успешные платежи.")
plt.show()

### График 2. Из чего сложилось изменение выручки

Каскад изменений, а не уровней: при оси от нуля шаги в несколько миллионов на фоне 50 млн были бы почти не видны.

In [ ]:
# Функция каскадной диаграммы лежит в src/prime/factors.py.
from prime.factors import waterfall


steps = effects.rename({"active": "активные\nподписки", "per_sub_day": "платежей\nна подписку",
                        "price": "цена\nплатежа"}) / 1000
ax = waterfall(steps, title=f"Цена дала {effects['price'] / 1000:+,.0f} тыс. ₽ в месяц "
                            f"из {total_delta / 1000:+,.0f} тыс. прироста".replace(",", " "))
ax.set_ylabel("тыс. ₽ в нормальный месяц")
ax.set_xlabel("Январь–февраль → март–июль 2026, выручка на день с данными × 30,4. "
              "Порядок подстановок — слева направо; тарифы сложены.", fontsize=9, color="#6B7280")
plt.show()

### График 3. Отток по тарифам

In [ ]:
wide = churn.pivot_table(index="month", columns="tariff", values="churn", aggfunc="sum") * 100
hike_x = (wide.index < PRICE_CHANGE[:7]).sum() - 0.5

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
for t, color, shift in (("Расширенный", GRAY, 10), ("Базовый", ACCENT, -10)):
    ax.plot(wide.index, wide[t], color=color, marker="o", linewidth=2.5 if color == ACCENT else 1.5)
    # подпись у конца линии; сдвиг вверх-вниз, чтобы подписи не легли друг на друга
    ax.annotate(t, (len(wide) - 1, wide[t].iloc[-1]), xytext=(8, shift), textcoords="offset points",
                color=color, va="center")
ax.axvline(hike_x, color="#111827", linestyle="--", linewidth=0.8)
ax.set_ylim(bottom=0)
ax.set_xlim(right=len(wide) + 0.5)
style(ax, f"После подорожания отток «Базового» изменился на {change['Базовый'] * 100:+.1f} п.п., "
          f"«Расширенного» — на {change['Расширенный'] * 100:+.1f} п.п.",
      "закончились в месяце, % от активных на начало",
      "Месяц по UTC, пробные подписки исключены. Пунктир — 1 марта 2026. Источник: prime.subscriptions.")
plt.show()

### График 4. Доля отмен «слишком дорого»

Над столбиками — сколько отмен с указанной причиной было в месяце.

In [ ]:
reasons = churn.groupby(["month", "period"], as_index=False)[["with_reason", "too_expensive"]].sum()
reasons["share"] = reasons["too_expensive"] / reasons["with_reason"] * 100
by_period = churn.groupby("period")[["too_expensive", "with_reason"]].sum()
pre_share = by_period.loc["до", "too_expensive"] / by_period.loc["до", "with_reason"]
post_share = by_period.loc["после", "too_expensive"] / by_period.loc["после", "with_reason"]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
bars = ax.bar(reasons["month"], reasons["share"],
              color=[ACCENT if p == "после" else GRAY for p in reasons["period"]], zorder=2)
ax.bar_label(bars, labels=[f"n={n}" for n in reasons["with_reason"]], padding=2, fontsize=8)
ax.margins(y=0.15)
style(ax, f"Доля отмен «слишком дорого»: {pre_share:.0%} до подорожания → {post_share:.0%} после",
      "too_expensive, % отмен с причиной",
      "Месяц отмены по UTC; красным — после 1 марта 2026. Источник: prime.subscriptions.cancel_reason.")
plt.show()

### График 5. Доля «Расширенного» среди активных подписок

In [ ]:
ext = share["Расширенный"]
ext = ext[ext.index < AFTER[1][:7]]

fig, ax = plt.subplots(figsize=(10, 4.6), layout="constrained")
ax.plot(ext.index, ext, color=ACCENT, marker="o", linewidth=2.5)
ax.axvline((ext.index < PRICE_CHANGE[:7]).sum() - 0.5, color="#111827", linestyle="--", linewidth=0.8)
ax.set_ylim(0, 100)      # шкала от 0 до 100 %, иначе сотые доли процента выглядят как скачок
style(ax, f"Доля «Расширенного» среди активных: {ext.iloc[0]:.1f} % → {ext.iloc[-1]:.1f} %",
      "% активных подписок",
      "Месяц по UTC. Пунктир — 1 марта 2026. Источник: prime.subscriptions.")
plt.show()

## 6. Вывод и рекомендация

Сводка чисел:

In [ ]:
summary = {
    "выручка нормального месяца, база -> после": f"{rub(level['база'])} -> {rub(level['после'])}"
                                                 f" ({total_delta / level['база']:+.1%})",
    "наивный прирост (средний месяц после − база)": rub(naive_delta),
    "влияние активных подписок": rub(effects["active"]),
    "влияние платежей на подписку": rub(effects["per_sub_day"]),
    "влияние цены платежа": rub(effects["price"]),
    "цена при других порядках, мин…макс": f"{rub(orders['price'].iloc[:-1].min())} … {rub(orders['price'].iloc[:-1].max())}",
    "средняя цена платежа: структура / цены": f"{by_structure:+.2f} ₽ / {by_price:+.2f} ₽",
    "отток «Базового», п.п.": f"{change['Базовый'] * 100:+.2f}",
    "отток «Расширенного», п.п.": f"{change['Расширенный'] * 100:+.2f}",
    "разница приростов, п.п.": f"{did * 100:+.2f} ± {did_ci * 100:.2f}",
    "доля too_expensive, до -> после": f"{pre_share:.1%} -> {post_share:.1%}",
    "раздутие выручки при счёте по пользователям": f"×{inflation:.2f}",
}
for k, v in summary.items():
    print(f"{k:<46} {v}")

### Записка

Что случилось: …

За счёт чего: …

Чего мы не знаем: …

Что делать: …